In [2]:
import os
import pandas as pd
from dbfread import DBF

# 모든 폴더, 파일 리스트 추출
def list_folder_recursive(path):
    """
    재귀적으로 경로 내의 모든 파일과 폴더(절대경로)를 리스트로 반환
    반환 항목: (root, dirs, files) 구조를 원하면 다른 형태로 변경 가능
    """
    results = []
    for root, dirs, files in os.walk(path):
        for d in dirs:
            results.append(os.path.join(root, d))
        for f in files:
            results.append(os.path.join(root, f))
    return results

shp_folder_path = r'Z:\111. 고속도로 정밀도로지도 검사(2025)\0. 검사결과\2. 속성검사(63종)_3차납품(01.16)'

err_dbf_list = []

all_items = list_folder_recursive(shp_folder_path)
# print(len(all_items), "items found")
all_items_df = pd.DataFrame(all_items, columns=['path'])
# all_items_df.to_csv('all_items_list.csv', index=False, encoding='utf-8-sig')

# na=False 로 NaN 처리, r'\\.dbf' 또는 r'\.dbf'로 점(.)을 이스케이프하여 정확히 '.dbf' 매칭
cond1 = (
    all_items_df['path'].str.contains('육안검사') &
    all_items_df['path'].str.contains('ERR') &
    all_items_df['path'].str.contains('.dbf')
)

all_items_df_filtered = all_items_df[cond1]
# all_items_df_filtered.to_csv('filtered_items_list.csv', index=False, encoding='utf-8-sig')
for i in range(len(all_items_df_filtered)):
    err_dbf_list.append(all_items_df_filtered.iloc[i, 0])

# 사업자 shp 전체 concat

# 여러개의 파일을 한번에 읽어 각각의 변수에 저장 - 딕셔너리로 진행
err_dbf_dict = {}
for i, path in enumerate(err_dbf_list):
    table = DBF(err_dbf_list[i], encoding='utf-8')
    err_dbf_dict[f'df_dbf_{i}'] = pd.DataFrame(iter(table))


df_dbf_all = pd.concat(err_dbf_dict.values(), ignore_index=True)
print("사업자 shp Total : ", df_dbf_all.shape)

# df_dbf_all.to_csv('ERR_dbf.csv', index=False, encoding='utf-8-sig')

사업자 shp Total :  (192, 5)


In [3]:
err_dbf_dict

{'df_dbf_0':   error_type error_expl Layer
 0          3  SUBTYPE오류   020
 1          4       형상오류   052
 2          4       형상오류   052
 3          4       형상오류   052
 4          4       형상오류   052
 5          4       형상오류   052
 6          4       형상오류   052
 7          4       형상오류   052
 8          4       형상오류   052,
 'df_dbf_1':   error_type error_expl Layer
 0          2       이정오류   008
 1          2       이정오류   008
 2          2       이정오류   050
 3          2       이정오류   062
 4          2       이정오류   062
 5          2       이정오류   052,
 'df_dbf_2':   error_type error_expl Layer
 0          2       이정오류   024
 1          2       이정오류   024
 2          2       이정오류   024
 3          2       이정오류   024,
 'df_dbf_3':   error_type error_expl Layer
 0          4       형상오류   052
 1          4       형상오류   052
 2          4       형상오류   052,
 'df_dbf_4':   error_type error_expl Layer QC_RESULT
 0          2       이정오류   046      오류아님,
 'df_dbf_5':   error_type error_expl Layer
 0  

In [11]:
err_dbf_list[96]

'Z:\\111. 고속도로 정밀도로지도 검사(2025)\\0. 검사결과\\2. 속성검사(63종)_2차납품(12.23)_송부파일\\26.01.07_송부(제작용)\\5_유원지리정보시스템\\영동선\\이천지사\\01. 251229_검사1\\육안검사\\이정매칭_부\\ERR_영동선_이천지사_이정.dbf'